In [ ]:
import pandas as pd
import json

In [ ]:
from eu_survey_correlation.surveys.ess_scraper import ESSCodebookParser

parser = ESSCodebookParser("data/surveys/ess/ESS8e02_3 codebook.html")
variable_df = parser.parse()

In [ ]:
variable_df

In [ ]:
dico_id_name_var = {
    row["variable_id"]: row["variable_name"] for _, row in variable_df.iterrows()
}

In [ ]:
value_df = pd.read_csv("data/surveys/ess/ESS8e02_3.csv")

In [ ]:
value_df.columns = [dico_id_name_var[col] for col in value_df.columns]

In [ ]:
value_df

In [ ]:
def check_row(
    row,
):
    print(row)
    for col in row.index:
        print(col)
        answer_value = variable_df[
            (variable_df["variable_name"] == col)
            & (variable_df["answer_id"] == row[col])
        ]
    row[col] = answer_value
    return row

In [ ]:
variable_df.variable_description.unique().tolist()

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

votes_df = pd.read_csv(
    "/Users/ugo/Documents/MH2D_projets/dawta/eu_survey_correlation/data/votes/votes.csv"
)

In [ ]:
import json
import os
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

output_file = "procedure_summaries.json"

# 1. Load existing data if it exists (so you don't re-scrape what you already have)
if os.path.exists(output_file):
    with open(output_file, "r") as f:
        procedure_summary_urls = json.load(f)
else:
    procedure_summary_urls = {}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

for procedure_ref in tqdm(procedure_references):
    # Skip if we already scraped this reference
    if procedure_ref in procedure_summary_urls:
        continue

    try:
        response = requests.get(
            f"https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference={procedure_ref}",
            headers=headers,
            timeout=10,
        )
        soup = BeautifulSoup(response.content, "html.parser")

        summary_url = None
        # Use a more robust selector to find the table
        section = soup.find(id="section3")

        if section:
            rows = section.find_all("tr")
            for row in rows:
                cols = row.find_all("td")
                if len(cols) >= 4:
                    event_name = cols[1].get_text(strip=True)
                    if "Decision by Parliament" in event_name:
                        link = cols[3].find("a", href=True)
                        if link:
                            href = link["href"]
                            summary_url = (
                                f"https://oeil.europarl.europa.eu{href}"
                                if href.startswith("/")
                                else href
                            )
                            break

        # Store the result (even if None, so we know we checked it)
        procedure_summary_urls[procedure_ref] = summary_url

        # 2. Save to JSON after EACH successful iteration
        with open(output_file, "w") as f:
            json.dump(procedure_summary_urls, f, indent=4)

    except Exception as e:
        print(f"\nError scraping {procedure_ref}: {e}")
        continue

In [ ]:
import json

In [ ]:
summary_url = "https://oeil.europarl.europa.eu/oeil/en/document-summary?id=1600759"

In [ ]:
response = requests.get(
    summary_url,
    timeout=10,
)
soup = BeautifulSoup(response.content, "html.parser")

In [ ]:
from bs4 import BeautifulSoup


def extract_clean_summary(html_content):
    """Parses the summary page and returns formatted text."""
    soup = BeautifulSoup(html_content, "html.parser")
    content_div = soup.find("div", class_="es_product-content")

    if not content_div:
        return ""

    paragraphs = content_div.find_all("p")
    clean_paragraphs = [
        p.get_text(" ", strip=True) for p in paragraphs if p.get_text(strip=True)
    ]

    return "\n\n".join(clean_paragraphs)


# Usage with your HTML:
# clean_text = extract_clean_summary(response.content)
# print(clean_text)

In [ ]:
Summary = extract_clean_summary(response.content)

In [ ]:
final_df = pd.read_csv(
    "/Users/ugo/Documents/MH2D_projets/dawta/eu_survey_correlation/data/votes/procedure_summaries_simplified.csv"
)

In [ ]:
final_df.simplified_summary

In [ ]:
final_df.summary_text.values[0]

In [ ]:
# --- CONFIGURATION ---
from pathlib import Path
from eu_survey_correlation.simplifier import Simplifier

DATA_DIR = Path("data")

SIMPLIFIER_CACHE = DATA_DIR / "cache" / "simplified_text.json"

In [ ]:
df = pd.read_csv(
    "/Users/ugo/Documents/MH2D_projets/dawta/eu_survey_correlation/data/surveys/all_survey_questions.csv"
)
# 2. Initialize your Simplifier
# M1 Pro hint: Mistral is great, but 'phi3' or 'llama3' are also very fast on Apple Silicon
simplifier = Simplifier(model="mistral", cache_path=SIMPLIFIER_CACHE)

# 3. Run the simplification
# We use your existing 'VOTE_SUMMARY_PROMPT'
df_simplified = simplifier.simplify_dataframe(
    df=df,
    text_column="question_en",
    output_column="question_clean",
    prompt_template=Simplifier.SURVEY_QUESTION_PROMPT,
    concurrency=10,  # Keep it low for local LLMs so your Mac doesn't lag
)

In [ ]:
df